# Cross-Model Evaluation

This notebook compares category labels, clustering labels, and factor-model estimates using summary statistics, Kruskal-Wallis tests, and partial $R^2$ calculations.

Category labels are sourced from `nft_metadata_base.category`. For category-based evaluation, rows with category values `unknown` and `vague` are excluded.

In [ ]:
import numpy as np
import pandas as pd
import bigframes.pandas as bpd
import statsmodels.api as sm
from scipy.stats import kruskal

PROJECT_ID = "<YOUR_GCP_PROJECT_ID>"
BASE_DATASET_ID = "<YOUR_BASE_BIGQUERY_DATASET>"
ANALYTICS_DATASET_ID = "<YOUR_ANALYTICS_BIGQUERY_DATASET>"

CLUSTERING_MASTER_TABLE = f"{PROJECT_ID}.{ANALYTICS_DATASET_ID}.clustering_master"
CLUSTERING_RESULT_TABLE = f"{PROJECT_ID}.{ANALYTICS_DATASET_ID}.clustering_result"
NFT_METADATA_BASE_TABLE = f"{PROJECT_ID}.{BASE_DATASET_ID}.nft_metadata_base"
BETA_ALPHA_GLOBAL_TABLE = f"{PROJECT_ID}.{ANALYTICS_DATASET_ID}.beta_alpha_estimates"
BETA_ALPHA_REGIME_TABLE = f"{PROJECT_ID}.{ANALYTICS_DATASET_ID}.beta_alpha_estimates_by_regime"
WEEKLY_RETURNS_TABLE = f"{PROJECT_ID}.{ANALYTICS_DATASET_ID}.weekly_returns_vw"
REGIME_LABELS_TABLE = f"{PROJECT_ID}.{ANALYTICS_DATASET_ID}.regime_labels"


In [ ]:
def load_cross_model_input() -> pd.DataFrame:
    sql = f"""
    WITH base AS (
        SELECT
            collection,
            first_trade_regime AS regime_class_struct,
            unique_holders, unique_holder_ratio, top10_share, holder_hhi, buyer_hhi, seller_hhi, longterm_holder_ratio,
            trading_days, active_weeks, trading_week_span, trades, daily_trades, buyer_seller_ratio,
            p25_price, median_price, p75_price, stddev_price, diff_price,
            description_length, has_project_url, has_twitter, has_discord, has_instagram, has_telegram,
            is_erc721, is_erc1155, has_erc2981, royalty_fee_percent
        FROM `{CLUSTERING_MASTER_TABLE}`
    ),
    cl AS (
        SELECT collection, cluster, distance_to_centroid
        FROM `{CLUSTERING_RESULT_TABLE}`
    ),
    cat AS (
        SELECT collection, category
        FROM `{NFT_METADATA_BASE_TABLE}`
    ),
    ab_global AS (
        SELECT
            collection,
            alpha_hat AS alpha_global,
            beta_market_hat AS beta_mkt_global,
            beta_fx_hat AS beta_fx_global,
            r2 AS r2_global,
            n_obs AS n_obs_global
        FROM `{BETA_ALPHA_GLOBAL_TABLE}`
    ),
    ab_regime AS (
        SELECT *
        FROM (
            SELECT
                collection,
                REPLACE(regime_class, '-', '_') AS regime,
                alpha_hat,
                beta_market_hat AS beta_mkt,
                beta_fx_hat AS beta_fx,
                r2,
                n_obs
            FROM `{BETA_ALPHA_REGIME_TABLE}`
        )
        PIVOT (
            ANY_VALUE(alpha_hat) AS alpha,
            ANY_VALUE(beta_mkt) AS beta_mkt,
            ANY_VALUE(beta_fx) AS beta_fx,
            ANY_VALUE(r2) AS r2,
            ANY_VALUE(n_obs) AS n_obs
            FOR regime IN ('pre_boom', 'boom', 'post_boom')
        )
    )
    SELECT
        b.collection,
        cat.category,
        cl.cluster,
        cl.distance_to_centroid,
        b.regime_class_struct,
        b.unique_holders, b.unique_holder_ratio, b.top10_share, b.holder_hhi, b.buyer_hhi, b.seller_hhi, b.longterm_holder_ratio,
        b.trading_days, b.active_weeks, b.trading_week_span, b.trades, b.daily_trades, b.buyer_seller_ratio,
        b.p25_price, b.median_price, b.p75_price, b.stddev_price, b.diff_price,
        b.description_length, b.has_project_url, b.has_twitter, b.has_discord, b.has_instagram, b.has_telegram,
        b.is_erc721, b.is_erc1155, b.has_erc2981, b.royalty_fee_percent,
        g.alpha_global, g.beta_mkt_global, g.beta_fx_global, g.r2_global, g.n_obs_global,
        r.alpha_pre_boom, r.alpha_boom, r.alpha_post_boom,
        r.beta_mkt_pre_boom, r.beta_mkt_boom, r.beta_mkt_post_boom,
        r.beta_fx_pre_boom, r.beta_fx_boom, r.beta_fx_post_boom,
        r.r2_pre_boom, r.r2_boom, r.r2_post_boom,
        r.n_obs_pre_boom, r.n_obs_boom, r.n_obs_post_boom
    FROM base b
    LEFT JOIN cl ON b.collection = cl.collection
    LEFT JOIN cat ON b.collection = cat.collection
    LEFT JOIN ab_global g ON b.collection = g.collection
    LEFT JOIN ab_regime r ON b.collection = r.collection
    ORDER BY b.collection
    """
    return bpd.read_gbq(sql).to_pandas()


df_eval = load_cross_model_input()
df_eval_valid_cat = df_eval[df_eval["category"].notna() & (~df_eval["category"].isin(["unknown", "vague"]))].copy()

print("Loaded cross-model evaluation table:", df_eval.shape)
print("Rows with valid category labels:", df_eval_valid_cat.shape)
print(df_eval.head())


In [ ]:
def make_tidy_alpha_beta(df: pd.DataFrame) -> pd.DataFrame:
    records = []
    regimes = ["pre_boom", "boom", "post_boom"]
    params = ["alpha", "beta_mkt", "beta_fx"]

    for _, row in df.iterrows():
        collection = row["collection"]
        category = row.get("category")
        cluster = row.get("cluster")

        for regime in regimes:
            n_obs = row.get(f"n_obs_{regime}")
            for param in params:
                value = row.get(f"{param}_{regime}")
                records.append({
                    "collection": collection,
                    "param": param,
                    "regime": regime,
                    "value": value,
                    "category": category,
                    "cluster": cluster,
                    "n_obs": n_obs,
                })

    return pd.DataFrame(records)


def summarize_by_group(df: pd.DataFrame, group_col: str, value_cols: list[str]) -> pd.DataFrame:
    rows = []
    for group_value, g in df.groupby(group_col, dropna=False):
        row = {group_col: group_value, "n_collections": len(g)}
        for col in value_cols:
            s = pd.to_numeric(g[col], errors="coerce").dropna()
            row[f"{col}_mean"] = s.mean() if len(s) else np.nan
            row[f"{col}_median"] = s.median() if len(s) else np.nan
            row[f"{col}_std"] = s.std() if len(s) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def summarize_by_two_groups(df: pd.DataFrame, group_cols: list[str], value_cols: list[str]) -> pd.DataFrame:
    rows = []
    for keys, g in df.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = {group_cols[i]: keys[i] for i in range(len(group_cols))}
        row["n_collections"] = len(g)
        for col in value_cols:
            s = pd.to_numeric(g[col], errors="coerce").dropna()
            row[f"{col}_mean"] = s.mean() if len(s) else np.nan
            row[f"{col}_median"] = s.median() if len(s) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


df_tidy = make_tidy_alpha_beta(df_eval)
df_tidy_valid_cat = df_tidy[df_tidy["category"].notna() & (~df_tidy["category"].isin(["unknown", "vague"]))].copy()

factor_cols_global = ["alpha_global", "beta_mkt_global", "beta_fx_global", "r2_global"]
cat_summary = summarize_by_group(df_eval_valid_cat, "category", factor_cols_global)
cluster_summary = summarize_by_group(df_eval, "cluster", factor_cols_global)
cat_cluster_summary = summarize_by_two_groups(df_eval_valid_cat, ["category", "cluster"], factor_cols_global)

print("\n=== Factor summary by category ===")
print(cat_summary.round(4))

print("\n=== Factor summary by cluster ===")
print(cluster_summary.round(4))

print("\n=== Factor summary by category × cluster ===")
print(cat_cluster_summary.round(4))


In [ ]:
def kruskal_by_group(df: pd.DataFrame, group_col: str, value_col: str = "value"):
    group_keys = sorted([g for g in df[group_col].unique() if pd.notna(g)])
    if len(group_keys) < 2:
        return None

    groups = []
    valid_keys = []
    for g in group_keys:
        vals = pd.to_numeric(df.loc[df[group_col] == g, value_col], errors="coerce").dropna().to_numpy(dtype=float)
        if len(vals) > 0:
            groups.append(vals)
            valid_keys.append(g)

    if len(groups) < 2:
        return None

    h_stat, p_value = kruskal(*groups)
    return {"H": float(h_stat), "p": float(p_value), "k": int(len(valid_keys))}


def run_unified_kruskal_comparison(df_tidy: pd.DataFrame, param: str) -> pd.DataFrame:
    rows = []
    df_param = df_tidy[df_tidy["param"] == param].copy()

    for regime in ["pre_boom", "boom", "post_boom"]:
        d = df_param[df_param["regime"] == regime].copy()
        d = d.dropna(subset=["value"])

        cat_res = kruskal_by_group(d, "category", "value")
        clu_res = kruskal_by_group(d, "cluster", "value")

        row = {"regime": regime, "n_obs": int(len(d)), "h_cat": None, "p_cat": None, "k_cat": None, "h_clu": None, "p_clu": None, "k_clu": None}
        if cat_res is not None:
            row.update({"h_cat": cat_res["H"], "p_cat": cat_res["p"], "k_cat": cat_res["k"]})
        if clu_res is not None:
            row.update({"h_clu": clu_res["H"], "p_clu": clu_res["p"], "k_clu": clu_res["k"]})
        rows.append(row)

    return pd.DataFrame(rows)


def format_unified_kruskal_results(df_kw: pd.DataFrame) -> pd.DataFrame:
    out = df_kw.copy()
    out["regime"] = out["regime"].str.replace("_", "-", regex=False)
    for col in ["h_cat", "h_clu"]:
        out[col] = out[col].map(lambda x: None if pd.isna(x) else round(float(x), 3))
    for col in ["p_cat", "p_clu"]:
        out[col] = out[col].map(lambda x: None if pd.isna(x) else f"{float(x):.2e}")
    return out


for param in ["alpha", "beta_mkt", "beta_fx"]:
    df_kw = run_unified_kruskal_comparison(
        df_tidy_valid_cat if param == "alpha" else df_tidy_valid_cat,
        param=param,
    )
    print(f"\n=== Unified Kruskal-Wallis comparison: {param} ===")
    print(format_unified_kruskal_results(df_kw))


In [ ]:
def compute_partial_r2(df: pd.DataFrame, min_obs: int = 30):
    work = df.copy()
    for col in ["nft_return", "market_return", "fx_return"]:
        work[col] = pd.to_numeric(work[col], errors="coerce")
    work = work.replace([np.inf, -np.inf], np.nan)
    work = work.dropna(subset=["nft_return", "market_return", "fx_return"])

    if len(work) < min_obs:
        return None, None, None

    y = work["nft_return"].astype(float)
    x_res = sm.add_constant(work[["market_return"]].astype(float))
    fit_res = sm.OLS(y, x_res).fit()
    x_full = sm.add_constant(work[["market_return", "fx_return"]].astype(float))
    fit_full = sm.OLS(y, x_full).fit()
    return fit_res.rsquared, fit_full.rsquared, fit_full.rsquared - fit_res.rsquared


def summarize_partial_r2_by_category(df_all: pd.DataFrame) -> pd.DataFrame:
    categories = sorted(df_all["category"].dropna().unique())
    regimes = ["pre_boom", "boom", "post_boom"]
    rows = []
    for category in categories:
        for regime in regimes:
            sub = df_all[(df_all["category"] == category) & (df_all["regime"] == regime)]
            r2_mkt, r2_full, r2_fx = compute_partial_r2(sub)
            rows.append({"category": category, "regime": regime, "r2_market": r2_mkt, "r2_full": r2_full, "partial_r2_fx": r2_fx})
    return pd.DataFrame(rows).dropna()


def summarize_partial_r2_by_cluster(df_all: pd.DataFrame) -> pd.DataFrame:
    clusters = sorted([c for c in df_all["cluster"].unique() if pd.notna(c)])
    regimes = ["pre_boom", "boom", "post_boom"]
    rows = []
    for cluster in clusters:
        for regime in regimes:
            sub = df_all[(df_all["cluster"] == cluster) & (df_all["regime"] == regime)]
            r2_mkt, r2_full, r2_fx = compute_partial_r2(sub)
            rows.append({"cluster": cluster, "regime": regime, "r2_market": r2_mkt, "r2_full": r2_full, "partial_r2_fx": r2_fx})
    return pd.DataFrame(rows).dropna()


df_cat_r2 = summarize_partial_r2_by_category(df_all=df_eval_valid_cat)
df_cluster_r2 = summarize_partial_r2_by_cluster(df_all=df_eval)

print("\n=== Partial R^2 by category and regime ===")
print(df_cat_r2.round(5))
print("\n=== Partial R^2 by cluster and regime ===")
print(df_cluster_r2.round(5))

print("\n=== Category pivot: partial R^2 of FX factor ===")
print(df_cat_r2.pivot(index="category", columns="regime", values="partial_r2_fx").reindex(columns=["pre_boom", "boom", "post_boom"]).round(4))

print("\n=== Cluster pivot: partial R^2 of FX factor ===")
print(df_cluster_r2.pivot(index="cluster", columns="regime", values="partial_r2_fx").reindex(columns=["pre_boom", "boom", "post_boom"]).round(4))
